In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import os
import json
import random

from PIL import Image, ImageDraw
import google.generativeai as genai
from PIL import Image, ImageDraw

In [20]:
IMAGE_DIR = "/content/drive/MyDrive/[Thesis] Improved_transformer/img"
RESULTS_DIR = "/content/drive/MyDrive/[Thesis] Improved_transformer/results"
BASELINE_DIR = "/content/drive/MyDrive/[Thesis] Improved_transformer/bbox_indicator_baseline/outputs/bbox_indicator_visual_pilot_lora/test_results"

PROPOSED_FILE = os.path.join(RESULTS_DIR, "[proposed_method]test_predictions_with_metrics.json")
BASELINE_FILE = os.path.join(BASELINE_DIR, "test_predictions_with_metrics.json")
VLM_POOL_FILE = os.path.join(RESULTS_DIR, "vlm_evaluation_pool.json")
RESULTS_DIR = "/content/drive/MyDrive/[Thesis] Improved_transformer/results"
OUTPUT_JSONL = os.path.join(RESULTS_DIR, "vlm_judge_results.jsonl")

SAMPLE_SIZE = 500

In [21]:
def normalize_key(img_id, reg_id):
    clean_reg = str(reg_id).replace("region_", "")
    return f"{str(img_id)}_{clean_reg}"

In [22]:
with open(PROPOSED_FILE, "r", encoding="utf-8") as f:
    prop_data = json.load(f)

with open(BASELINE_FILE, "r", encoding="utf-8") as f:
    base_data = json.load(f)

base_dict = {normalize_key(r['image_id'], r['region_id']): r for r in base_data}

In [23]:
prop_data, base_dict

([{'image_id': 2322119,
   'region_id': 'region_4676336',
   'bbox': [261, 175, 36, 100],
   'gt_en': 'There is a home plate here',
   'gt_vi': 'Có một tấm home plate ở đây',
   'prediction': 'home plate is white',
   'bleu4': 21.4441,
   'meteor': 44.0613,
   'rougeL': 40.0,
   'cider': 247.9114,
   'clip_score': 0.0,
   'chair_s': 0.0,
   'chair_i': 0,
   'ref_siglip_score': 0.0},
  {'image_id': 2322119,
   'region_id': 'region_4676337',
   'bbox': [156, 105, 39, 77],
   'gt_en': 'There is a baseball bat',
   'gt_vi': 'Có một cây gậy bóng chày',
   'prediction': 'a black baseball bat',
   'bleu4': 27.5348,
   'meteor': 52.1542,
   'rougeL': 66.6667,
   'cider': 259.6107,
   'clip_score': 0.0,
   'chair_s': 0.0,
   'chair_i': 0,
   'ref_siglip_score': 0.0},
  {'image_id': 2322119,
   'region_id': 'region_4676338',
   'bbox': [244, 38, 18, 24],
   'gt_en': 'There is a helmet here',
   'gt_vi': 'Có một chiếc mũ bảo hiểm ở đây',
   'prediction': 'the helmet is black',
   'bleu4': 14.794,

In [24]:
merged_data = []
for prop in prop_data:
    key = normalize_key(prop['image_id'], prop['region_id'])
    base = base_dict.get(key)

    if base:
        merged_data.append({
            "image_id": prop['image_id'],
            "region_id": prop['region_id'],
            "bbox": prop['bbox'],
            "gt_en": prop['gt_en'],
            "pred_baseline": base.get('prediction', ''),
            "pred_proposed": prop.get('prediction', '')
        })

print(f"Total merged samples: {len(merged_data)}")

Total merged samples: 36486


In [25]:
if not os.path.exists(VLM_POOL_FILE):
    random.seed(42)
    eval_pool = random.sample(merged_data, min(SAMPLE_SIZE, len(merged_data)))
    with open(VLM_POOL_FILE, "w", encoding="utf-8") as f:
        json.dump(eval_pool, f, indent=2)
    print(f"Created new evaluation pool with {len(eval_pool)} samples: {VLM_POOL_FILE}")
else:
    with open(VLM_POOL_FILE, "r", encoding="utf-8") as f:
        eval_pool = json.load(f)
    print(f"Loaded existing evaluation pool with {len(eval_pool)} samples.")

Loaded existing evaluation pool with 500 samples.


In [26]:
API_KEY =
genai.configure(api_key=API_KEY)
MODEL_NAME = "gemini-3.1-flash-lite"

In [33]:
START_IDX = 193
END_IDX = 500

with open(VLM_POOL_FILE, "r", encoding="utf-8") as f:
    eval_pool = json.load(f)

processed_keys = set()
if os.path.exists(OUTPUT_JSONL):
    with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            processed_keys.add(f"{data['image_id']}_{data['region_id']}")

In [29]:
def get_image_with_bbox(img_path, bbox):
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    x, y, w, h = bbox
    draw.rectangle([x, y, x + w, y + h], outline="red", width=4)
    return img

In [30]:
system_prompt = """You are an expert evaluator for Dense Image Captioning.
You are given an image with a specific region highlighted by a thick RED bounding box.
You are provided with a Ground Truth caption and two candidate predictions (Model A and Model B).

Task: Evaluate which model better describes the specific region enclosed by the RED bounding box.
Rules:
1. The description MUST accurately reflect the content inside the RED box.
2. Penalize hallucination (mentioning objects/actions not present in the box).
3. Penalize generic/global descriptions if the box highlights a specific local detail.
4. If both are equally good or equally bad, choose Tie.

Output your response strictly in JSON format: {"winner": "A" or "B" or "Tie", "reason": "short explanation"}"""

model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    system_instruction=system_prompt,
    generation_config={"response_mime_type": "application/json", "temperature": 0.0}
)

print(f"{START_IDX} - {END_IDX}")

174 - 194


In [31]:
import time

In [34]:
for i in range(START_IDX, min(END_IDX, len(eval_pool))):
    sample = eval_pool[i]
    img_id = sample['image_id']
    reg_id = sample['region_id']
    key = f"{img_id}_{reg_id}"

    if key in processed_keys:
        continue

    img_path = None
    for ext in (".jpg", ".jpeg", ".png"):
        p = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
        if os.path.exists(p):
            img_path = p
            break

    if not img_path:
        print(f"[{i}] Missing image {img_id}")
        continue

    img_with_bbox = get_image_with_bbox(img_path, sample['bbox'])

    is_proposed_A = random.choice([True, False])
    pred_A = sample['pred_proposed'] if is_proposed_A else sample['pred_baseline']
    pred_B = sample['pred_baseline'] if is_proposed_A else sample['pred_proposed']

    user_prompt = f"Ground Truth: {sample['gt_en']}\nModel A: {pred_A}\nModel B: {pred_B}"

    try:
        response = model.generate_content([user_prompt, img_with_bbox])
        content = json.loads(response.text)

        vlm_winner = content.get("winner", "Tie")
        if vlm_winner == "A":
            actual_winner = "Proposed" if is_proposed_A else "Baseline"
        elif vlm_winner == "B":
            actual_winner = "Baseline" if is_proposed_A else "Proposed"
        else:
            actual_winner = "Tie"

        result_record = {
            "image_id": img_id,
            "region_id": reg_id,
            "winner": actual_winner,
            "reason": content.get("reason", "")
        }

        with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
            f.write(json.dumps(result_record) + "\n")

        print(f"[{i}] Processed {key} | Winner: {actual_winner}")
        time.sleep(3)

    except Exception as e:
        print(f"[{i}] Error on {key}: {e}")
        time.sleep(5)

[194] Processed 179_region_4937772 | Winner: Tie
[195] Processed 2380677_region_1484750 | Winner: Proposed
[196] Processed 2330588_region_4071930 | Winner: Proposed
[197] Processed 2357409_region_2792016 | Winner: Tie
[198] Processed 2354588_region_2926466 | Winner: Proposed
[199] Processed 2319043_region_5439251 | Winner: Proposed
[200] Processed 2356741_region_2823243 | Winner: Tie
[201] Processed 2341400_region_3555609 | Winner: Baseline
[202] Processed 2413899_region_82619 | Winner: Baseline
[203] Processed 2319717_region_4391334 | Winner: Proposed
[204] Processed 1160256_region_6036723 | Winner: Tie
[205] Processed 2329499_region_4123684 | Winner: Proposed
[206] Processed 2351440_region_3076536 | Winner: Proposed
[207] Processed 2320238_region_5431972 | Winner: Proposed
[208] Processed 2355656_region_2875410 | Winner: Tie
[209] Processed 2358619_region_2734211 | Winner: Proposed
[210] Processed 2323582_region_4606227 | Winner: Proposed
[211] Processed 2326924_region_4246501 | Winn

In [35]:
RESULTS_DIR = "/content/drive/MyDrive/[Thesis] Improved_transformer/results"
OUTPUT_JSONL = os.path.join(RESULTS_DIR, "vlm_judge_results.jsonl")

if not os.path.exists(OUTPUT_JSONL):
    print("No results found.")
else:
    results = []
    with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            results.append(json.loads(line))

    total = len(results)
    wins = sum(1 for r in results if r["winner"] == "Proposed")
    losses = sum(1 for r in results if r["winner"] == "Baseline")
    ties = sum(1 for r in results if r["winner"] == "Tie")

    print("VLM Evaluation Summary")
    print(f"Total Samples Assessed: {total}")
    print(f"Proposed Wins : {wins} ({(wins/total)*100:.2f}%)")
    print(f"Baseline Wins : {losses} ({(losses/total)*100:.2f}%)")
    print(f"Ties          : {ties} ({(ties/total)*100:.2f}%)")

VLM Evaluation Summary
Total Samples Assessed: 501
Proposed Wins : 315 (62.87%)
Baseline Wins : 67 (13.37%)
Ties          : 119 (23.75%)
